# AgeLens — 02 Preprocessing, Harmonization, and Diagnostic Phenotypic Age

This notebook implements the AgeLens V1 preprocessing sequence:

```text
01 native-unit ingestion output
→ native-unit cross-cycle bridging
→ formula-unit conversion
→ fasting-subsample definition
→ complete-case exclusion
→ diagnostic-only formula variants
```

## Placement

```text
nhanes/
└── notebooks/
    ├── 00_setup_agelens.ipynb
    ├── 01_data_ingestion.ipynb
    └── 02_data_preprocessing.ipynb
```

## Required input

```text
nhanes/data/interim/nhanes_2015_2018_ingested_native.parquet
```

## Scientific safeguards

- Bridging is applied **before** unit conversion.
- `LBXGLU`, not `LBXSGL`, is used.
- The fasting subsample is identified before incidental missingness is assessed.
- No imputation is performed.
- hs-CRP values outside the validated bridging-comparison range are flagged and are not silently mixed into the primary harmonized CRP column.
- CRP must be strictly positive before the natural logarithm.
- ALP must be strictly positive before the log-Deming bridging equation.
- The erratum and Supplement formula constants are kept as internally consistent pairs.
- EG-004 creatinine adjustment is not invented or silently applied.
- `RIDAGEYR == 80` remains flagged; no unsupported correction is applied.
- No unapproved age restriction is silently introduced.
- Every Phenotypic Age value produced here is labeled **diagnostic only**, not a final scientific result.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 190)

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


## 1. Locate the project and load the controlled input

In [2]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'. "
        "Run this notebook from inside the nhanes project."
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}\n"
        "Run notebooks/00_setup_agelens.ipynb first."
    )

CONFIG: dict[str, Any] = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
PROCESSED_ROOT = PROJECT_ROOT / CONFIG["paths"]["processed_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]

for path in [INTERIM_ROOT, PROCESSED_ROOT, TABLES_ROOT, LOGS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    INTERIM_ROOT / "nhanes_2015_2018_ingested_native.parquet"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input not found: {INPUT_PATH}\n"
        "Run notebooks/01_data_ingestion.ipynb first."
    )

data = pd.read_parquet(INPUT_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_PATH}")
print(f"Rows: {len(data):,}")
print(f"Columns: {len(data.columns)}")


Project root: <PROJECT_ROOT>
Input: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_ingested_native.parquet
Rows: 19,225
Columns: 24


## 2. Validate configuration and input schema

In [3]:
required_config_sections = {
    "project",
    "governance",
    "nhanes",
    "source_variables",
    "pipeline",
    "unit_conversions",
    "bridging",
    "formula",
    "paths",
}
missing_config_sections = required_config_sections - set(CONFIG)
if missing_config_sections:
    raise ValueError(
        f"Configuration is missing sections: {sorted(missing_config_sections)}"
    )

assert CONFIG["pipeline"]["missing_data_policy"] == "complete_case"
assert CONFIG["pipeline"]["imputation_allowed"] is False
assert CONFIG["pipeline"]["bridge_before_unit_conversion"] is True
assert CONFIG["source_variables"]["glucose"] == "LBXGLU"
assert CONFIG["formula"]["xb_coefficients"]["glucose"] == 0.1953
assert CONFIG["project"]["final_scientific_results_allowed"] is False

cycle_column = CONFIG["nhanes"]["cycle_column"]
reference_cycle = CONFIG["bridging"]["reference_cycle"]
bridge_cycle = CONFIG["bridging"]["apply_to_cycle"]

required_columns = {
    "SEQN",
    cycle_column,
    "RIDAGEYR",
    "age_topcoded",
    "RIAGENDR",
    "RIDRETH3",
    "WTSAF2YR",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "LBXSAL",
    "LBXSCR",
    "LBXSAPSI",
    "LBXGLU",
    "LBXHSCRP",
    "LBXLYPCT",
    "LBXMCVSI",
    "LBXRDW",
    "LBXWBCSI",
}

missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise ValueError(f"Input is missing required columns: {missing_columns}")

if "LBXSGL" in data.columns:
    raise ValueError(
        "LBXSGL is present in the controlled input. "
        "The primary pipeline must use LBXGLU from GLU_I/J."
    )

XPT_IBM_ZERO_SENTINEL = np.float64(
    5.397605346934028e-79
)

controlled_numeric_columns = sorted(
    required_columns
    - {
        "SEQN",
        cycle_column,
    }
)

sentinel_hits = []

for column in controlled_numeric_columns:
    values = pd.to_numeric(
        data[column],
        errors="coerce",
    )
    hit_count = int(
        values.eq(XPT_IBM_ZERO_SENTINEL).sum()
    )

    if hit_count:
        sentinel_hits.append(
            {
                "column": column,
                "count": hit_count,
            }
        )

if sentinel_hits:
    raise RuntimeError(
        "Controlled input contains unnormalized "
        "pandas/XPORT IBM-zero sentinel values. "
        "Rerun the corrected 01_data_ingestion.ipynb. "
        f"Hits: {sentinel_hits}"
    )

if data.duplicated([cycle_column, "SEQN"]).any():
    raise ValueError("Duplicate cycle + SEQN combinations found.")

observed_cycles = set(data[cycle_column].dropna().unique())
expected_cycles = set(CONFIG["nhanes"]["cycles"])
if observed_cycles != expected_cycles:
    raise ValueError(
        f"Unexpected cycles. Observed={sorted(observed_cycles)}, "
        f"expected={sorted(expected_cycles)}"
    )

print("✅ Configuration and input schema verified.")


✅ Configuration and input schema verified.


## 3. Preserve raw native-unit variables and define traceability flags

No raw source column is overwritten. All bridging and conversion outputs receive new names.


In [4]:
working = data.copy()

raw_native_columns = {
    "LBXSAL": "albumin_raw_g_dL",
    "LBXSCR": "creatinine_raw_mg_dL",
    "LBXSAPSI": "alp_raw_U_L",
    "LBXGLU": "glucose_raw_mg_dL",
    "LBXHSCRP": "crp_raw_mg_L",
    "LBXLYPCT": "lymphocyte_percent_raw",
    "LBXMCVSI": "mcv_raw_fL",
    "LBXRDW": "rdw_raw_percent",
    "LBXWBCSI": "wbc_raw_1000cells_uL",
    "RIDAGEYR": "chronological_age_years",
}

for source, target in raw_native_columns.items():
    working[target] = pd.to_numeric(working[source], errors="coerce")

working["age_topcoded"] = working["chronological_age_years"].eq(
    CONFIG["nhanes"]["age_topcode_value"]
)
working["age_below_20"] = (
    working["chronological_age_years"].notna()
    & working["chronological_age_years"].lt(20)
)

working["in_fasting_subsample"] = (
    pd.to_numeric(working["WTSAF2YR"], errors="coerce").fillna(0) > 0
)

working["glucose_measured"] = working["glucose_raw_mg_dL"].notna()
working["fasting_glucose_incidentally_missing"] = (
    working["in_fasting_subsample"]
    & ~working["glucose_measured"]
)
working["glucose_structurally_missing_outside_fasting_subsample"] = (
    ~working["in_fasting_subsample"]
    & ~working["glucose_measured"]
)

print("Fasting-subsample counts:")
display(
    working.groupby(cycle_column, observed=True).agg(
        total_rows=("SEQN", "size"),
        fasting_subsample=("in_fasting_subsample", "sum"),
        glucose_measured=("glucose_measured", "sum"),
        fasting_glucose_incidentally_missing=(
            "fasting_glucose_incidentally_missing",
            "sum",
        ),
        age_below_20=("age_below_20", "sum"),
        age_topcoded=("age_topcoded", "sum"),
    )
)


Fasting-subsample counts:


,total_rows,fasting_subsample,glucose_measured,fasting_glucose_incidentally_missing,age_below_20,age_topcoded
NHANES_CYCLE,,,,,,
2015_2016,9971,2743,2972,0,4252,376
2017_2018,9254,2711,2891,0,3685,427


## 4. Native-unit cross-cycle bridging

2015–2016 is adjusted to the 2017–2018 reference scale.

### hs-CRP boundary policy

The validated comparison range is:

- 2015–2016 raw DxC value: `≤ 23 mg/L`
- 2017–2018 reference-scale value: `≤ 20 mg/L`

Above-range values are retained in their raw source columns and flagged. They are set to missing in the dedicated `crp_harmonized_mg_L` column so adjusted and unadjusted values are not silently blended into one variable presented as harmonized.


In [5]:
is_bridge_cycle = working[cycle_column].eq(bridge_cycle)
is_reference_cycle = working[cycle_column].eq(reference_cycle)

if not (is_bridge_cycle | is_reference_cycle).all():
    raise ValueError("Rows outside the configured two-cycle scope were found.")

# Input validity flags. Values are not clipped or winsorized.
working["invalid_albumin_nonpositive"] = (
    working["albumin_raw_g_dL"].notna()
    & working["albumin_raw_g_dL"].le(0)
)
working["invalid_creatinine_nonpositive"] = (
    working["creatinine_raw_mg_dL"].notna()
    & working["creatinine_raw_mg_dL"].le(0)
)
working["invalid_glucose_nonpositive"] = (
    working["glucose_raw_mg_dL"].notna()
    & working["glucose_raw_mg_dL"].le(0)
)
working["invalid_crp_nonpositive"] = (
    working["crp_raw_mg_L"].notna()
    & working["crp_raw_mg_L"].le(0)
)
working["invalid_alp_nonpositive"] = (
    working["alp_raw_U_L"].notna()
    & working["alp_raw_U_L"].le(0)
)
working["invalid_lymphocyte_range"] = (
    working["lymphocyte_percent_raw"].notna()
    & ~working["lymphocyte_percent_raw"].between(0, 100)
)
working["invalid_mcv_nonpositive"] = (
    working["mcv_raw_fL"].notna()
    & working["mcv_raw_fL"].le(0)
)
working["invalid_rdw_nonpositive"] = (
    working["rdw_raw_percent"].notna()
    & working["rdw_raw_percent"].le(0)
)
working["invalid_wbc_nonpositive"] = (
    working["wbc_raw_1000cells_uL"].notna()
    & working["wbc_raw_1000cells_uL"].le(0)
)

# Albumin: Cobas = 0.9581 * DxC660i - 0.0108
working["albumin_harmonized_g_dL"] = working["albumin_raw_g_dL"]
albumin_bridge_mask = (
    is_bridge_cycle
    & working["albumin_raw_g_dL"].notna()
    & ~working["invalid_albumin_nonpositive"]
)
working.loc[
    albumin_bridge_mask,
    "albumin_harmonized_g_dL",
] = (
    0.9581
    * working.loc[albumin_bridge_mask, "albumin_raw_g_dL"]
    - 0.0108
)
working["albumin_bridge_applied"] = albumin_bridge_mask

# Creatinine: Cobas = 0.9515 * DxC660i + 0.06608
working["creatinine_harmonized_mg_dL"] = (
    working["creatinine_raw_mg_dL"]
)
creatinine_bridge_mask = (
    is_bridge_cycle
    & working["creatinine_raw_mg_dL"].notna()
    & ~working["invalid_creatinine_nonpositive"]
)
working.loc[
    creatinine_bridge_mask,
    "creatinine_harmonized_mg_dL",
] = (
    0.9515
    * working.loc[
        creatinine_bridge_mask,
        "creatinine_raw_mg_dL",
    ]
    + 0.06608
)
working["creatinine_bridge_applied"] = creatinine_bridge_mask
working["eg004_creatinine_training_era_adjustment_applied"] = False

# ALP: Cobas = 10 ** (0.9986 * log10(DxC660i) + 0.04288)
working["alp_harmonized_U_L"] = working["alp_raw_U_L"]
alp_bridge_mask = (
    is_bridge_cycle
    & working["alp_raw_U_L"].notna()
    & ~working["invalid_alp_nonpositive"]
)
working.loc[
    alp_bridge_mask,
    "alp_harmonized_U_L",
] = 10 ** (
    0.9986
    * np.log10(
        working.loc[alp_bridge_mask, "alp_raw_U_L"]
    )
    + 0.04288
)
working["alp_bridge_applied"] = alp_bridge_mask

# hs-CRP comparable range.
working["crp_above_validated_bridge_range"] = (
    (
        is_bridge_cycle
        & working["crp_raw_mg_L"].gt(23.0)
    )
    | (
        is_reference_cycle
        & working["crp_raw_mg_L"].gt(20.0)
    )
)

working["crp_within_validated_bridge_range"] = (
    working["crp_raw_mg_L"].notna()
    & ~working["invalid_crp_nonpositive"]
    & ~working["crp_above_validated_bridge_range"]
)

working["crp_harmonized_mg_L"] = np.nan

crp_bridge_mask = (
    is_bridge_cycle
    & working["crp_within_validated_bridge_range"]
)
working.loc[
    crp_bridge_mask,
    "crp_harmonized_mg_L",
] = (
    0.8695
    * working.loc[crp_bridge_mask, "crp_raw_mg_L"]
    + 0.2954
)

crp_reference_mask = (
    is_reference_cycle
    & working["crp_within_validated_bridge_range"]
)
working.loc[
    crp_reference_mask,
    "crp_harmonized_mg_L",
] = working.loc[
    crp_reference_mask,
    "crp_raw_mg_L",
]

working["crp_bridge_applied"] = crp_bridge_mask
working["crp_reference_value_used"] = crp_reference_mask
working["crp_excluded_from_primary_harmonized_column"] = (
    working["crp_raw_mg_L"].notna()
    & ~working["crp_within_validated_bridge_range"]
)

bridge_counts = working.groupby(cycle_column, observed=True).agg(
    albumin_bridge_applied=("albumin_bridge_applied", "sum"),
    creatinine_bridge_applied=("creatinine_bridge_applied", "sum"),
    alp_bridge_applied=("alp_bridge_applied", "sum"),
    crp_bridge_applied=("crp_bridge_applied", "sum"),
    crp_above_validated_range=(
        "crp_above_validated_bridge_range",
        "sum",
    ),
    crp_nonpositive=("invalid_crp_nonpositive", "sum"),
    alp_nonpositive=("invalid_alp_nonpositive", "sum"),
)

display(bridge_counts)


,albumin_bridge_applied,creatinine_bridge_applied,alp_bridge_applied,crp_bridge_applied,crp_above_validated_range,crp_nonpositive,alp_nonpositive
NHANES_CYCLE,,,,,,,
2015_2016,6256,6255,6255,7718,149,0,0
2017_2018,0,0,0,0,132,0,0


## 5. Convert to formula-required units

In [6]:
# Pre-bridging formula-unit columns: retained for Validation Check 3.
working["albumin_prebridge_g_L"] = (
    working["albumin_raw_g_dL"] * 10.0
)
working["creatinine_prebridge_umol_L"] = (
    working["creatinine_raw_mg_dL"] * 88.4
)
working["glucose_mmol_L"] = (
    working["glucose_raw_mg_dL"] * 0.0555
)
working["crp_prebridge_mg_dL"] = (
    working["crp_raw_mg_L"] / 10.0
)

# Primary harmonized formula-unit columns.
working["albumin_harmonized_g_L"] = (
    working["albumin_harmonized_g_dL"] * 10.0
)
working["creatinine_harmonized_umol_L"] = (
    working["creatinine_harmonized_mg_dL"] * 88.4
)
working["crp_harmonized_mg_dL"] = (
    working["crp_harmonized_mg_L"] / 10.0
)

# Unchanged-unit biomarkers.
working["lymphocyte_percent"] = (
    working["lymphocyte_percent_raw"]
)
working["mcv_fL"] = working["mcv_raw_fL"]
working["rdw_percent"] = working["rdw_raw_percent"]
working["wbc_1000cells_uL"] = (
    working["wbc_raw_1000cells_uL"]
)

# CRP log: only strictly positive values.
working["log_crp_prebridge"] = np.nan
prebridge_positive_crp = working["crp_prebridge_mg_dL"].gt(0)
working.loc[
    prebridge_positive_crp,
    "log_crp_prebridge",
] = np.log(
    working.loc[
        prebridge_positive_crp,
        "crp_prebridge_mg_dL",
    ]
)

working["log_crp_harmonized"] = np.nan
harmonized_positive_crp = working[
    "crp_harmonized_mg_dL"
].gt(0)
working.loc[
    harmonized_positive_crp,
    "log_crp_harmonized",
] = np.log(
    working.loc[
        harmonized_positive_crp,
        "crp_harmonized_mg_dL",
    ]
)

# Ensure invalid values do not reach formula-ready columns.
working.loc[
    working["invalid_albumin_nonpositive"],
    ["albumin_prebridge_g_L", "albumin_harmonized_g_L"],
] = np.nan
working.loc[
    working["invalid_creatinine_nonpositive"],
    [
        "creatinine_prebridge_umol_L",
        "creatinine_harmonized_umol_L",
    ],
] = np.nan
working.loc[
    working["invalid_glucose_nonpositive"],
    "glucose_mmol_L",
] = np.nan
working.loc[
    working["invalid_alp_nonpositive"],
    "alp_harmonized_U_L",
] = np.nan
working.loc[
    working["invalid_lymphocyte_range"],
    "lymphocyte_percent",
] = np.nan
working.loc[
    working["invalid_mcv_nonpositive"],
    "mcv_fL",
] = np.nan
working.loc[
    working["invalid_rdw_nonpositive"],
    "rdw_percent",
] = np.nan
working.loc[
    working["invalid_wbc_nonpositive"],
    "wbc_1000cells_uL",
] = np.nan

print("✅ Native-unit bridging and formula-unit conversion completed.")


✅ Native-unit bridging and formula-unit conversion completed.


## 6. Define the fasting analytic frame and complete-case indicators

Fasting glucose absence outside the fasting subsample is treated as structural missingness. Complete-case assessment of the other biomarkers is evaluated only after restricting to participants with a positive fasting-subsample weight.

No participant is imputed.


In [7]:
shared_formula_columns = [
    "glucose_mmol_L",
    "lymphocyte_percent",
    "mcv_fL",
    "rdw_percent",
    "wbc_1000cells_uL",
    "chronological_age_years",
]

prebridge_formula_columns = [
    "albumin_prebridge_g_L",
    "creatinine_prebridge_umol_L",
    *shared_formula_columns[:1],
    "log_crp_prebridge",
    *shared_formula_columns[1:],
    "alp_raw_U_L",
]

harmonized_formula_columns = [
    "albumin_harmonized_g_L",
    "creatinine_harmonized_umol_L",
    *shared_formula_columns[:1],
    "log_crp_harmonized",
    *shared_formula_columns[1:],
    "alp_harmonized_U_L",
]

# The eight non-glucose biomarkers used for missingness audits among fasting participants.
remaining_eight_harmonized_columns = [
    "albumin_harmonized_g_L",
    "creatinine_harmonized_umol_L",
    "log_crp_harmonized",
    "lymphocyte_percent",
    "mcv_fL",
    "rdw_percent",
    "alp_harmonized_U_L",
    "wbc_1000cells_uL",
]

working["complete_case_prebridge"] = (
    working["in_fasting_subsample"]
    & working[prebridge_formula_columns].notna().all(axis=1)
)

working["complete_case_harmonized"] = (
    working["in_fasting_subsample"]
    & working[harmonized_formula_columns].notna().all(axis=1)
)

# Identical sample for pre/post bridging comparison.
working["complete_case_bridge_comparison"] = (
    working["complete_case_prebridge"]
    & working["complete_case_harmonized"]
)

working["complete_case_harmonized_no_topcode"] = (
    working["complete_case_harmonized"]
    & ~working["age_topcoded"]
)

working["complete_case_harmonized_age20plus"] = (
    working["complete_case_harmonized"]
    & working["chronological_age_years"].ge(20)
)

sample_flow = (
    working.groupby(cycle_column, observed=True)
    .agg(
        total_rows=("SEQN", "size"),
        fasting_subsample=("in_fasting_subsample", "sum"),
        glucose_measured=("glucose_measured", "sum"),
        complete_case_prebridge=("complete_case_prebridge", "sum"),
        complete_case_harmonized=(
            "complete_case_harmonized",
            "sum",
        ),
        bridge_comparison_sample=(
            "complete_case_bridge_comparison",
            "sum",
        ),
        harmonized_no_topcode=(
            "complete_case_harmonized_no_topcode",
            "sum",
        ),
        harmonized_age20plus=(
            "complete_case_harmonized_age20plus",
            "sum",
        ),
        crp_above_validated_range=(
            "crp_above_validated_bridge_range",
            "sum",
        ),
    )
    .reset_index()
)

display(sample_flow)


,NHANES_CYCLE,total_rows,fasting_subsample,glucose_measured,complete_case_prebridge,complete_case_harmonized,bridge_comparison_sample,harmonized_no_topcode,harmonized_age20plus,crp_above_validated_range
0,2015_2016,9971,2743,2972,2701,2645,2645,2524,2181,149
1,2017_2018,9254,2711,2891,2638,2578,2578,2427,2186,132


## 7. Missingness audit among fasting-subsample participants

In [8]:
missingness_records = []

for cycle, frame in working.loc[
    working["in_fasting_subsample"]
].groupby(cycle_column, observed=True):
    for variable in remaining_eight_harmonized_columns:
        missingness_records.append(
            {
                "cycle": cycle,
                "variable": variable,
                "fasting_subsample_n": int(len(frame)),
                "missing_n": int(frame[variable].isna().sum()),
                "missing_percent": float(
                    frame[variable].isna().mean() * 100
                ),
            }
        )

missingness_audit = pd.DataFrame(missingness_records)

display(
    missingness_audit.pivot(
        index="variable",
        columns="cycle",
        values="missing_percent",
    ).round(2)
)


cycle,2015_2016,2017_2018
variable,,
albumin_harmonized_g_L,0.80,2.18
alp_harmonized_U_L,0.84,2.25
creatinine_harmonized_umol_L,0.80,2.21
log_crp_harmonized,2.99,4.61
lymphocyte_percent,0.47,0.11
mcv_fL,0.47,0.00
rdw_percent,0.47,0.00
wbc_1000cells_uL,0.47,0.00


## 8. Diagnostic Phenotypic Age calculation

The mortality score is computed with `expm1` for numerical stability. Phenotypic Age uses the algebraically equivalent log-hazard expression:

```text
-ln(1 - M) = 1.51714 × exp(xb) / 0.0076927
```

This avoids a numerical failure when floating-point rounding makes `M` equal to exactly 1.

Both published constant pairs are calculated:

- Erratum pair: `141.50 / 0.09165`
- Supplement pair: `141.50225 / 0.090165`

They are never mixed.


In [9]:
coefficients = CONFIG["formula"]["xb_coefficients"]
mortality_constants = CONFIG["formula"]["mortality_constants"]
formula_variants = CONFIG["formula"]["diagnostic_variants"]

required_open_gaps = {"EG-004", "EG-010", "EG-014"}
open_gaps = set(CONFIG["governance"]["open_core_evidence_gaps"])
if not required_open_gaps.issubset(open_gaps):
    raise ValueError(
        "The diagnostic governance guard expected EG-004, EG-010, "
        "and EG-014 to remain explicitly recorded."
    )


def calculate_diagnostic_phenoage(
    frame: pd.DataFrame,
    *,
    complete_mask: pd.Series,
    albumin_column: str,
    creatinine_column: str,
    glucose_column: str,
    log_crp_column: str,
    lymphocyte_column: str,
    mcv_column: str,
    rdw_column: str,
    alp_column: str,
    wbc_column: str,
    age_column: str,
    output_prefix: str,
) -> None:
    idx = frame.index[complete_mask]

    xb = pd.Series(np.nan, index=frame.index, dtype="float64")

    xb.loc[idx] = (
        coefficients["intercept"]
        + coefficients["albumin"] * frame.loc[idx, albumin_column]
        + coefficients["creatinine"] * frame.loc[idx, creatinine_column]
        + coefficients["glucose"] * frame.loc[idx, glucose_column]
        + coefficients["log_crp"] * frame.loc[idx, log_crp_column]
        + coefficients["lymphocyte_percent"]
        * frame.loc[idx, lymphocyte_column]
        + coefficients["mcv"] * frame.loc[idx, mcv_column]
        + coefficients["rdw"] * frame.loc[idx, rdw_column]
        + coefficients["alp"] * frame.loc[idx, alp_column]
        + coefficients["wbc"] * frame.loc[idx, wbc_column]
        + coefficients["chronological_age"]
        * frame.loc[idx, age_column]
    )

    log_hazard = (
        np.log(
            mortality_constants["numerator"]
            / mortality_constants["denominator"]
        )
        + xb
    )

    # M is retained for diagnostics. Clipping affects only the displayed
    # mortality score at extreme floating-point limits, not Phenotypic Age.
    hazard_for_m = np.exp(log_hazard.clip(lower=-745, upper=709))
    mortality_score = -np.expm1(-hazard_for_m)

    frame[f"{output_prefix}_xb"] = xb
    frame[f"{output_prefix}_log_hazard"] = log_hazard
    frame[f"{output_prefix}_mortality_score"] = mortality_score

    for variant_name, variant in formula_variants.items():
        frame[
            f"{output_prefix}_phenoage_{variant_name}_years"
        ] = (
            variant["intercept"]
            + (
                np.log(-variant["multiplier"])
                + log_hazard
            )
            / variant["denominator"]
        )


calculate_diagnostic_phenoage(
    working,
    complete_mask=working["complete_case_prebridge"],
    albumin_column="albumin_prebridge_g_L",
    creatinine_column="creatinine_prebridge_umol_L",
    glucose_column="glucose_mmol_L",
    log_crp_column="log_crp_prebridge",
    lymphocyte_column="lymphocyte_percent",
    mcv_column="mcv_fL",
    rdw_column="rdw_percent",
    alp_column="alp_raw_U_L",
    wbc_column="wbc_1000cells_uL",
    age_column="chronological_age_years",
    output_prefix="prebridge",
)

calculate_diagnostic_phenoage(
    working,
    complete_mask=working["complete_case_harmonized"],
    albumin_column="albumin_harmonized_g_L",
    creatinine_column="creatinine_harmonized_umol_L",
    glucose_column="glucose_mmol_L",
    log_crp_column="log_crp_harmonized",
    lymphocyte_column="lymphocyte_percent",
    mcv_column="mcv_fL",
    rdw_column="rdw_percent",
    alp_column="alp_harmonized_U_L",
    wbc_column="wbc_1000cells_uL",
    age_column="chronological_age_years",
    output_prefix="harmonized",
)

working["harmonized_constant_pair_difference_years"] = (
    working["harmonized_phenoage_supplement_years"]
    - working["harmonized_phenoage_erratum_years"]
)

working["harmonized_phenoage_minus_age_erratum_years"] = (
    working["harmonized_phenoage_erratum_years"]
    - working["chronological_age_years"]
)

working["harmonized_phenoage_minus_age_supplement_years"] = (
    working["harmonized_phenoage_supplement_years"]
    - working["chronological_age_years"]
)

working["diagnostic_only"] = True
working["final_scientific_result"] = False

print("✅ Diagnostic formula variants calculated.")


✅ Diagnostic formula variants calculated.


## 9. Internal formula and output checks

The stable expression is checked directly for every complete case.

The staged expression reconstructs the cumulative hazard from the stored mortality score `M`. Because `M` loses floating-point precision when it becomes extremely close to 1, staged-versus-stable equivalence is checked only where cumulative hazard is at most 10. This tests the algebra without treating expected floating-point information loss as a scientific formula error.


In [10]:
# Formula values must exist exactly where their complete-case masks are true.
for prefix, mask_column in [
    ("prebridge", "complete_case_prebridge"),
    ("harmonized", "complete_case_harmonized"),
]:
    expected = working[mask_column]

    for variant in formula_variants:
        output_column = f"{prefix}_phenoage_{variant}_years"

        if working.loc[expected, output_column].isna().any():
            raise RuntimeError(
                f"{output_column} is missing inside its complete-case sample."
            )

        if working.loc[~expected, output_column].notna().any():
            raise RuntimeError(
                f"{output_column} exists outside its complete-case sample."
            )

# Independent internal check of the stable erratum expression.
erratum = formula_variants["erratum"]

expected_stable_erratum = (
    erratum["intercept"]
    + (
        np.log(-erratum["multiplier"])
        + working["harmonized_log_hazard"]
    )
    / erratum["denominator"]
)

stable_formula_mask = working["complete_case_harmonized"]

if not np.allclose(
    expected_stable_erratum.loc[stable_formula_mask],
    working.loc[
        stable_formula_mask,
        "harmonized_phenoage_erratum_years",
    ],
    rtol=1e-12,
    atol=1e-12,
):
    raise RuntimeError(
        "The stored stable Phenotypic Age values do not match "
        "the configured erratum expression."
    )

# Verify the stable expression against the staged published expression.
#
# When the cumulative hazard is large, M is so close to 1 that storing M
# as float64 loses information. Reconstructing -ln(1-M) from that rounded
# M is then numerically unstable even though the stable formula is correct.
# The staged comparison is therefore restricted to a range where M retains
# adequate floating-point precision.
MAX_HAZARD_FOR_STAGED_CHECK = 10.0

check_mask = (
    working["complete_case_harmonized"]
    & working["harmonized_log_hazard"].le(
        np.log(MAX_HAZARD_FOR_STAGED_CHECK)
    )
    & working["harmonized_mortality_score"].gt(0)
    & working["harmonized_mortality_score"].lt(1)
)

if not check_mask.any():
    raise RuntimeError(
        "No numerically reliable observations were available "
        "for the staged-formula equivalence check."
    )

mortality_score_for_check = working.loc[
    check_mask,
    "harmonized_mortality_score",
]

staged_inner = (
    erratum["multiplier"]
    * np.log1p(-mortality_score_for_check)
)

if not staged_inner.gt(0).all():
    raise RuntimeError(
        "The staged Phenotypic Age logarithm received a "
        "non-positive inner value."
    )

staged_erratum = (
    erratum["intercept"]
    + np.log(staged_inner)
    / erratum["denominator"]
)

stable_erratum = working.loc[
    check_mask,
    "harmonized_phenoage_erratum_years",
]

absolute_difference = (staged_erratum - stable_erratum).abs()
maximum_absolute_difference = float(absolute_difference.max())

if not np.allclose(
    staged_erratum,
    stable_erratum,
    rtol=1e-9,
    atol=1e-8,
):
    raise RuntimeError(
        "Stable formula does not match the staged published formula "
        "within the numerically reliable comparison range. "
        f"Maximum absolute difference: "
        f"{maximum_absolute_difference:.12g} years."
    )

print(
    "Staged/stable equivalence check passed on "
    f"{int(check_mask.sum()):,} observations; "
    f"maximum absolute difference = "
    f"{maximum_absolute_difference:.3e} years."
)

# No infinite diagnostic outputs.
diagnostic_numeric_columns = [
    column
    for column in working.columns
    if (
        "_phenoage_" in column
        or column.endswith("_xb")
        or column.endswith("_log_hazard")
    )
]

for column in diagnostic_numeric_columns:
    values = working[column].dropna().to_numpy()
    if not np.isfinite(values).all():
        raise RuntimeError(f"Non-finite values found in {column}.")

print("✅ Formula equivalence and finite-value checks passed.")

Staged/stable equivalence check passed on 5,204 observations; maximum absolute difference = 3.411e-13 years.
✅ Formula equivalence and finite-value checks passed.


## 10. Diagnostic summaries

In [11]:
diagnostic_summary_records = []

for cycle, frame in working.loc[
    working["complete_case_harmonized"]
].groupby(cycle_column, observed=True):
    for variant in formula_variants:
        column = f"harmonized_phenoage_{variant}_years"

        diagnostic_summary_records.append(
            {
                "cycle": cycle,
                "formula_variant": variant,
                "n": int(frame[column].notna().sum()),
                "mean": float(frame[column].mean()),
                "std": float(frame[column].std()),
                "min": float(frame[column].min()),
                "median": float(frame[column].median()),
                "max": float(frame[column].max()),
                "age_correlation_full_unweighted": float(
                    frame[[column, "chronological_age_years"]]
                    .corr(method="pearson")
                    .iloc[0, 1]
                ),
                "age_correlation_excluding_topcoded_unweighted": float(
                    frame.loc[
                        ~frame["age_topcoded"],
                        [column, "chronological_age_years"],
                    ]
                    .corr(method="pearson")
                    .iloc[0, 1]
                ),
            }
        )

diagnostic_summary = pd.DataFrame(diagnostic_summary_records)
display(diagnostic_summary.round(4))

constant_difference_summary = (
    working.loc[
        working["complete_case_harmonized"],
        [
            cycle_column,
            "harmonized_constant_pair_difference_years",
        ],
    ]
    .groupby(cycle_column, observed=True)
    .agg(["count", "mean", "std", "min", "median", "max"])
)

print("Supplement minus erratum diagnostic difference:")
display(constant_difference_summary.round(4))


,cycle,formula_variant,n,mean,std,min,median,max,age_correlation_full_unweighted,age_correlation_excluding_topcoded_unweighted
0,2015_2016,erratum,2645,44.9466,23.1965,2.8138,44.4917,159.2171,0.9394,0.9343
1,2015_2016,supplement,2645,43.3586,23.5785,0.5319,42.8963,159.5111,0.9394,0.9343
2,2017_2018,erratum,2578,47.2358,23.5144,-2.6540,47.6948,191.8045,0.9387,0.9307
3,2017_2018,supplement,2578,45.6855,23.9017,-5.0259,46.1521,192.6352,0.9387,0.9307


Supplement minus erratum diagnostic difference:


harmonized_constant_pair_difference_years                                        
                                                 count    mean     std     min  median     max
NHANES_CYCLE                                                                                  
2015_2016                                         2645 -1.5880  0.3820 -2.2819 -1.5955  0.2940
2017_2018                                         2578 -1.5503  0.3873 -2.3719 -1.5427  0.8308

## 11. Pre/post bridging audit on an identical sample

These summaries are descriptive and unweighted. Formal cross-cycle testing belongs to the Validation Protocol and must use the NHANES survey design.


In [12]:
bridge_comparison = working.loc[
    working["complete_case_bridge_comparison"]
].copy()

bridge_effect_records = []

bridge_pairs = {
    "albumin_g_L": (
        "albumin_prebridge_g_L",
        "albumin_harmonized_g_L",
    ),
    "creatinine_umol_L": (
        "creatinine_prebridge_umol_L",
        "creatinine_harmonized_umol_L",
    ),
    "crp_mg_dL": (
        "crp_prebridge_mg_dL",
        "crp_harmonized_mg_dL",
    ),
    "alp_U_L": (
        "alp_raw_U_L",
        "alp_harmonized_U_L",
    ),
    "phenoage_erratum_years": (
        "prebridge_phenoage_erratum_years",
        "harmonized_phenoage_erratum_years",
    ),
    "phenoage_supplement_years": (
        "prebridge_phenoage_supplement_years",
        "harmonized_phenoage_supplement_years",
    ),
}

for cycle, frame in bridge_comparison.groupby(
    cycle_column,
    observed=True,
):
    for measure, (pre_column, post_column) in bridge_pairs.items():
        difference = frame[post_column] - frame[pre_column]

        bridge_effect_records.append(
            {
                "cycle": cycle,
                "measure": measure,
                "n": int(difference.notna().sum()),
                "pre_mean": float(frame[pre_column].mean()),
                "post_mean": float(frame[post_column].mean()),
                "mean_post_minus_pre": float(difference.mean()),
                "median_post_minus_pre": float(difference.median()),
                "min_post_minus_pre": float(difference.min()),
                "max_post_minus_pre": float(difference.max()),
            }
        )

bridge_effect_audit = pd.DataFrame(bridge_effect_records)
display(bridge_effect_audit.round(4))


,cycle,measure,n,pre_mean,post_mean,mean_post_minus_pre,median_post_minus_pre,min_post_minus_pre,max_post_minus_pre
0,2015_2016,albumin_g_L,2645,43.3879,41.4619,-1.9260,-1.9097,-2.4544,-1.2812
1,2015_2016,creatinine_umol_L,2645,73.8686,76.1275,2.2588,2.4544,-37.9758,4.4695
2,2015_2016,crp_mg_dL,2645,0.2998,0.2902,-0.0096,0.0087,-0.2706,0.0285
3,2015_2016,alp_U_L,2645,80.9338,88.7675,7.8337,6.7103,2.2756,54.0488
4,2015_2016,phenoage_erratum_years,2645,43.5956,44.9466,1.3510,1.1698,-3.1724,3.6670
5,2015_2016,phenoage_supplement_years,2645,41.9854,43.3586,1.3732,1.1891,-3.2246,3.7274
6,2017_2018,albumin_g_L,2578,40.3860,40.3860,0.0000,0.0000,0.0000,0.0000
7,2017_2018,creatinine_umol_L,2578,76.6772,76.6772,0.0000,0.0000,0.0000,0.0000
8,2017_2018,crp_mg_dL,2578,0.2962,0.2962,0.0000,0.0000,0.0000,0.0000
9,2017_2018,alp_U_L,2578,87.8988,87.8988,0.0000,0.0000,0.0000,0.0000


## 12. Save transformed, diagnostic, and audit outputs

The all-participant interim file retains raw, bridged, standardized, flag, and diagnostic columns.

The processed complete-case files are convenience outputs for validation. They remain diagnostic and are not final release datasets.


In [13]:
written_files: list[Path] = []

all_participant_output = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)
working.to_parquet(all_participant_output, index=False)
written_files.append(all_participant_output)

harmonized_complete_case = working.loc[
    working["complete_case_harmonized"]
].copy()

harmonized_complete_case_output = (
    PROCESSED_ROOT
    / "agelens_harmonized_complete_case_diagnostic.parquet"
)
harmonized_complete_case.to_parquet(
    harmonized_complete_case_output,
    index=False,
)
written_files.append(harmonized_complete_case_output)

no_topcode_output = (
    PROCESSED_ROOT
    / "agelens_harmonized_complete_case_no_topcode_diagnostic.parquet"
)
working.loc[
    working["complete_case_harmonized_no_topcode"]
].to_parquet(no_topcode_output, index=False)
written_files.append(no_topcode_output)

age20plus_output = (
    PROCESSED_ROOT
    / "agelens_harmonized_complete_case_age20plus_diagnostic.parquet"
)
working.loc[
    working["complete_case_harmonized_age20plus"]
].to_parquet(age20plus_output, index=False)
written_files.append(age20plus_output)

bridge_comparison_output = (
    PROCESSED_ROOT
    / "agelens_bridge_comparison_complete_case_diagnostic.parquet"
)
bridge_comparison.to_parquet(
    bridge_comparison_output,
    index=False,
)
written_files.append(bridge_comparison_output)

sample_flow_path = TABLES_ROOT / "02_sample_flow.csv"
missingness_audit_path = (
    TABLES_ROOT / "02_fasting_sample_missingness_audit.csv"
)
bridge_effect_path = TABLES_ROOT / "02_bridge_effect_audit.csv"
diagnostic_summary_path = (
    TABLES_ROOT / "02_diagnostic_formula_summary.csv"
)
invalid_value_audit_path = (
    TABLES_ROOT / "02_invalid_value_audit.csv"
)

sample_flow.to_csv(sample_flow_path, index=False)
missingness_audit.to_csv(missingness_audit_path, index=False)
bridge_effect_audit.to_csv(bridge_effect_path, index=False)
diagnostic_summary.to_csv(diagnostic_summary_path, index=False)

invalid_flag_columns = [
    "invalid_albumin_nonpositive",
    "invalid_creatinine_nonpositive",
    "invalid_glucose_nonpositive",
    "invalid_crp_nonpositive",
    "invalid_alp_nonpositive",
    "invalid_lymphocyte_range",
    "invalid_mcv_nonpositive",
    "invalid_rdw_nonpositive",
    "invalid_wbc_nonpositive",
]

invalid_value_audit = (
    working.groupby(cycle_column, observed=True)[invalid_flag_columns]
    .sum()
    .reset_index()
)
invalid_value_audit.to_csv(invalid_value_audit_path, index=False)

written_files.extend(
    [
        sample_flow_path,
        missingness_audit_path,
        bridge_effect_path,
        diagnostic_summary_path,
        invalid_value_audit_path,
    ]
)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "02_data_preprocessing.ipynb",
    "input": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "row_count_all_participants": int(len(working)),
    "row_count_fasting_subsample": int(
        working["in_fasting_subsample"].sum()
    ),
    "row_count_complete_case_prebridge": int(
        working["complete_case_prebridge"].sum()
    ),
    "row_count_complete_case_harmonized": int(
        working["complete_case_harmonized"].sum()
    ),
    "row_count_complete_case_harmonized_no_topcode": int(
        working["complete_case_harmonized_no_topcode"].sum()
    ),
    "row_count_complete_case_harmonized_age20plus": int(
        working["complete_case_harmonized_age20plus"].sum()
    ),
    "bridge_direction": f"{bridge_cycle} -> {reference_cycle}",
    "bridge_before_unit_conversion": True,
    "xpt_ibm_zero_sentinel_guard_passed": True,
    "xpt_ibm_zero_sentinel_value": float(
        XPT_IBM_ZERO_SENTINEL
    ),
    "complete_case_policy": True,
    "imputation_applied": False,
    "crp_above_range_primary_policy": (
        "flag_and_set_harmonized_crp_missing; "
        "raw value retained separately"
    ),
    "age_restriction_applied_to_primary_complete_case": False,
    "age_below_20_flag_created": True,
    "age_topcode_correction_applied": False,
    "eg004_creatinine_adjustment_applied": False,
    "formula_variants": list(formula_variants),
    "formula_output_status": "diagnostic_only",
    "mortality_data_used": False,
    "survey_statistics_run": False,
    "final_scientific_results_allowed": False,
    "open_core_evidence_gaps": sorted(open_gaps),
    "outputs": [
        str(path.relative_to(PROJECT_ROOT))
        for path in written_files
    ],
}

metadata_path = (
    LOGS_ROOT / "02_data_preprocessing_metadata.json"
)
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
written_files.append(metadata_path)

print("Files written:")
for path in written_files:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")


Files written:
  - data\interim\nhanes_2015_2018_preprocessed_diagnostic.parquet
  - data\processed\agelens_harmonized_complete_case_diagnostic.parquet
  - data\processed\agelens_harmonized_complete_case_no_topcode_diagnostic.parquet
  - data\processed\agelens_harmonized_complete_case_age20plus_diagnostic.parquet
  - data\processed\agelens_bridge_comparison_complete_case_diagnostic.parquet
  - results\tables\02_sample_flow.csv
  - results\tables\02_fasting_sample_missingness_audit.csv
  - results\tables\02_bridge_effect_audit.csv
  - results\tables\02_diagnostic_formula_summary.csv
  - results\tables\02_invalid_value_audit.csv
  - logs\02_data_preprocessing_metadata.json


## 13. Reload verification

In [14]:
reloaded_all = pd.read_parquet(
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)
reloaded_complete = pd.read_parquet(
    PROCESSED_ROOT
    / "agelens_harmonized_complete_case_diagnostic.parquet"
)

assert len(reloaded_all) == len(working)
assert len(reloaded_complete) == int(
    working["complete_case_harmonized"].sum()
)
assert reloaded_complete[
    harmonized_formula_columns
].notna().all().all()
assert reloaded_complete[
    "harmonized_phenoage_erratum_years"
].notna().all()
assert reloaded_complete[
    "harmonized_phenoage_supplement_years"
].notna().all()
assert reloaded_complete["diagnostic_only"].all()
assert not reloaded_complete["final_scientific_result"].any()
assert "LBXSGL" not in reloaded_all.columns
assert not reloaded_all.duplicated(
    [cycle_column, "SEQN"]
).any()

print("✅ Preprocessing and diagnostic output verification passed.")
print("No imputation was performed.")
print("Mortality data were not used.")
print("Outputs are diagnostic only.")
print("Next step: validation against BioAge and survey-design checks.")


✅ Preprocessing and diagnostic output verification passed.
No imputation was performed.
Mortality data were not used.
Outputs are diagnostic only.
Next step: validation against BioAge and survey-design checks.
